# Model Results: Imbalance Strategy and Ensemble Comparison

Run `python train.py --data data/raw/creditcard.csv` from the repository root first. This notebook reads the saved later-time holdout predictions and makes project figures. It does not retrain models or modify the test set.

In [ ]:
from pathlib import Path
import json, math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, ConfusionMatrixDisplay
sns.set_theme(style='whitegrid')
root_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in root_candidates if (p/'reports/model_comparison.csv').exists()), None)
if ROOT is None: raise FileNotFoundError('Run train.py first; reports/model_comparison.csv is missing.')
metrics = pd.read_csv(ROOT/'reports/model_comparison.csv')
pred = pd.read_csv(ROOT/'reports/holdout_transactions.csv')
meta = json.loads((ROOT/'reports/run_metadata.json').read_text(encoding='utf-8'))
print('Holdout class counts (untouched):', meta['holdout_class_counts_untouched'])
print('Models fit:', metrics['model'].unique().tolist())
metrics.head()

## Metrics at equal review capacity

Recall at budget asks how much labeled fraud is inside the top-ranked alert slots. Precision at budget is the share of reviewed cases that have a fraud label. Average precision summarizes the precision-recall curve. A 99.8%+ accuracy score would mostly reflect the legitimate majority, so it is not the headline metric.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.barplot(data=metrics, x='alert_budget_pct', y='recall_at_budget_pct', hue='model', ax=axes[0])
axes[0].set_title('Fraud caught within alert budget'); axes[0].set_xlabel('Holdout share reviewed (%)'); axes[0].set_ylabel('Recall (%)')
sns.barplot(data=metrics, x='alert_budget_pct', y='precision_at_budget_pct', hue='model', ax=axes[1])
axes[1].set_title('Fraud share of reviewed alerts'); axes[1].set_xlabel('Holdout share reviewed (%)'); axes[1].set_ylabel('Precision (%)')
plt.tight_layout(); plt.show()

In [ ]:
display(metrics[['model','alert_budget_pct','alerts_reviewed','holdout_fraud','fraud_caught','accuracy_pct','precision_at_budget_pct','recall_at_budget_pct','f1_at_budget_pct','false_alerts','missed_fraud','average_precision','roc_auc']].sort_values(['alert_budget_pct','average_precision'], ascending=[True,False]).round(4))

## Precision-recall curves

These curves use the untouched chronological holdout. Average precision can be compared across models, but always report the holdout fraud count and class prevalence beside it.

In [ ]:
score_names = {'rf_smote':'Random Forest + SMOTE','rf_weighted':'Random Forest + class weights','xgb_smote':'XGBoost + SMOTE','xgb_weighted':'XGBoost + class weights'}
y_true = pred['Class'].astype(int).to_numpy()
plt.figure(figsize=(8,6))
for key, label in score_names.items():
    precision, recall, _ = precision_recall_curve(y_true, pred[f'score_{key}'])
    plt.plot(recall, precision, label=label)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-recall curves on later-time holdout'); plt.legend(); plt.show()

## Confusion matrix at the review budget

This is a ranking-based review list: for a chosen model and budget, flag the top-scoring K holdout transactions. The threshold is not tuned using the test labels; K is fixed by review capacity.

In [ ]:
model_key = 'rf_smote'
budget_fraction = 0.005
score = pred[f'score_{model_key}'].to_numpy()
k = max(1, math.ceil(len(pred) * budget_fraction))
flagged = score >= sorted(score, reverse=True)[k-1]
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, flagged, labels=[0,1])
print(f'Model={score_names[model_key]}, review budget={100*budget_fraction:.2f}%, alerts={k}')
ConfusionMatrixDisplay(cm, display_labels=['Legitimate','Fraud']).plot(cmap='Blues', values_format='d'); plt.title('Holdout confusion matrix at fixed alert budget'); plt.show()

## Feature importance and limits

Random Forest feature importance indicates reliance within this fitted model. It does not explain why a specific transaction is fraudulent, establish causality, or decode V1-V28.

In [ ]:
fi_path = ROOT/'reports/feature_importance.csv'
if fi_path.exists():
    fi = pd.read_csv(fi_path).head(15).sort_values('importance')
    fi.plot(kind='barh', x='feature', y='importance', legend=False, figsize=(8,6), title='Random Forest + SMOTE: top feature importances')
    plt.xlabel('Importance'); plt.tight_layout(); plt.show()

## Reporting checklist

- Give the exact temporal split sizes and fraud counts from `run_metadata.json`.
- Compare SMOTE to class weighting for each model, not just RF versus XGBoost.
- State that SMOTE was inside the training pipeline and the holdout was never resampled.
- Explain Random Forest is bagging and XGBoost is boosting.
- Do not say the CSV itself is overfit. Overfitting concerns model generalization; the old pipeline had data leakage and invalid evaluation.
- A single holdout comparison is initial evidence, not proof of production performance.